# `Notebook 10`: `Part 3 - Sparse Matrix Storage`
_Version 0.0.2_

*All of the header information is important. Please read it..*

**Topics number of exercises:** This problem builds on your knowledge of `Sparse Matrix Storage, Numpy, Scipy, Graph Networks, COO, CSR,`. It has **8** exercises numbered 0 to **7**. There are **24** available points. However to earn 100% the threshold is **24** points. (Therefore once you hit **24** points you can stop. There is no extra credit for exceeding this threshold.)

**Exercise ordering:** Each exercise builds logically on previous exercises but you may solve them in any order. That is if you can't solve an exercise you can still move on and try the next one. Use this to your advantage as the exercises are **not** necessarily ordered in terms of difficulty. Higher point values generally indicate more difficult exercises. 

**Demo cells:** Code cells starting with the comment `### Run Me!!!` load results from prior exercises applied to the entire data set and use those to build demo inputs. These must be run for subsequent demos to work properly but they do not affect the test cells. The data loaded in these cells may be rather large (at least in terms of human readability). You are free to print or otherwise use Python to explore them but we may not print them in the starter code.

**Debugging your code:** Right before each exercise test cell there is a block of text explaining the variables available to you for debugging. You may use these to test your code and can print/display them as needed (careful when printing large objects you may want to print the head or chunks of rows at a time).

**Exercise point breakdown:**


- Exercise 0 - : **3** point(s)

- Exercise 1 - : **3** point(s)

- Exercise 2 - : **3** point(s)

- Exercise 3 - : **3** point(s)

- Exercise 4 - : **3** point(s)

- Exercise 5 - : **3** point(s)

- Exercise 6 - : **3** point(s)

- Exercise 7 - : **3** point(s)


**Final reminders:** 

- Submit after **every exercise**
- Review the generated grade report after you submit to see what errors were returned
- Stay calm, skip problems as needed and take short breaks at your leisure

In [9]:
### Global imports
import dill
from cse6040_devkit import plugins, utils



In [10]:
import collections
import pandas as pd
import timeit
import numpy as np

### Preliminatry data and functions
**Matrix Functions**
- `sparse_matrix` function with nested default dictionaries
- `dense_vector` function which initalizes vector based on list passed or length

In [11]:
def sparse_matrix(base_type=float):
    """Returns a sparse matrix using nested default dictionaries."""
    from collections import defaultdict
    return defaultdict(lambda: defaultdict (base_type))

def dense_vector(init, base_type=float):
    """
    Returns a dense vector, either of a given length
    and initialized to 0 values or using a given list
    of initial values.
    """
    # Case 1: `init` is a list of initial values for the vector entries
    if type(init) is list:
        initial_values = init
        return [base_type(x) for x in initial_values]
    
    # Else, case 2: `init` is a vector length.
    assert type(init) is int
    return [base_type(0)] * init

def vector_keyed(keys=None, values=0, base_type=float):
    if keys is not None:
        if type(values) is not list:
            values = [base_type(values)] * len(keys)
        else:
            values = [base_type(v) for v in values]
        x = dict(zip(keys, values))
    else:
        x = {}
    return x

**Import Graph Network**
- Import data
- Build bidirectional source->target and target->source graph and store in `edges`
- Get counts of number of edges `num_edges` and number of verticies `num_verts`
- Build `id2name` and `name2id`

In [12]:
edges_raw = pd.read_csv('resource/asnlib/publicdata/UserEdges-1M.csv')
display(edges_raw.head ())
print("...\n`edges_raw` has {} entries.".format(len(edges_raw)))

,Source,Target
0,18kPq7GPye-YQ3LyKyAZPw,rpOyqD_893cqmDAtJLbdog
1,18kPq7GPye-YQ3LyKyAZPw,4U9kSBLuBDU391x6bxU-YA
2,18kPq7GPye-YQ3LyKyAZPw,fHtTaujcyKvXglE33Z5yIw
3,18kPq7GPye-YQ3LyKyAZPw,8J4IIYcqBlFch8T90N923A
4,18kPq7GPye-YQ3LyKyAZPw,wy6l_zUo7SN0qrvNRWgySw


...
`edges_raw` has 1000000 entries.


In [13]:
edges_raw_trans = pd.DataFrame({'Source': edges_raw['Target'],
                                'Target': edges_raw['Source']})
edges_raw_symm = pd.concat([edges_raw, edges_raw_trans])
# edges = edges_raw_symm.drop_duplicates()
#order edges
edges = edges_raw_symm.drop_duplicates().sort_values(['Source','Target'])

V_namesset = set(edges['Source'])
V_namesset.update(set(edges['Target']))
#order V_names
V_names=list(V_namesset)
V_names.sort()

num_edges = len(edges)
num_verts = len(V_names)
print("==> |V| == {}, |E| == {}".format(num_verts, num_edges))

==> |V| == 107456, |E| == 882640


In [14]:
id2name = {} # id2name[id] == name
name2id = {} # name2id[name] == id

for k, v in enumerate (V_names):
    # for debugging
    if k <= 5: print ("Name %s -> Vertex id %d" % (v, k))
    if k == 6: print ("...")
        
    id2name[k] = v
    name2id[v] = k

Name --0KsjlAThNWua2Pr4HStQ -> Vertex id 0
Name --0mI_q_0D1CdU4P_hoImQ -> Vertex id 1
Name --4fX3LBeXoE88gDTK6TKQ -> Vertex id 2
Name --65q1FpAL_UQtVZ2PTGew -> Vertex id 3
Name --7266Nwi6RXjKNNKeoFMQ -> Vertex id 4
Name --9HuvEtLhp21SeIEEltnA -> Vertex id 5
...


### Exercise 0: (3 points)
**spmv**  

**Your task:** define `spmv` as follows:

Create a function to compute $y \leftarrow A x$.

**Inputs**:
- `A`: sparse_matrix technically a collections.defaultdict
- `x`: dense_vector technically a list
- `num_rows`: int default None

**Return**: Return $y \leftarrow A x$
- `y`: dense_vector technically a list

**Hints**
- Recall you did a matrix-vector multiply in Notebook 10 Part 2. Adapt that logic for when `A` is now a sparse matrix.
- Conceptually, for all $i$, $y_{i} = \sum ^{j} A_{ij}x_{j}$


In [15]:
### Solution - Exercise 0  

def spmv(A: collections.defaultdict, x: list, num_rows:int=None) -> list:
    if num_rows is None:
        num_rows = max(A.keys()) + 1 #find the highest row number if num_rows value isn't provided
    y = dense_vector(num_rows) #dense_vector function which initalizes vector based on list passed or length
    
    #y = A*x
    #A deaultdict mapping row/col to values 
    
    #Iterate over sparse matrix A as nested dictionary
    for i, row_data in A.items(): #Iterate over each row index 'i' in the matrix
        for j, value in row_data.items(): # row_data is another dictionary: {column_index: value}
            if j < len(x):
                y[i] += value * x[j]  # Multiply the value at row i, col j by vector x at index j
                                     # and add it to the result for row i
  
    return y

##############################


    #m: number of rows
    #n: number of columns
    #matrix_list A: 1D list of size m*n (column-major)
    #vector_list X: 1D list of size n
    

### Demo function call
#   / 0.   -2.5   1.2 \   / 1. \   / -1.4 \
#   | 0.1   1.    0.  | * | 2. | = |  2.1 |
#   \ 6.   -1.    0.  /   \ 3. /   \  4.0 /
A = sparse_matrix()
A[0][1] = -2.5
A[0][2] = 1.2
A[1][0] = 0.1
A[1][1] = 1.
A[2][0] = 6.
A[2][1] = -1.

x = dense_vector ([1, 2, 3])
y0 = dense_vector ([-1.4, 2.1, 4.0])

print('Here is what the input data looks like:')
print(f'A={A}\nx={x}')
y = spmv(A, x)
print('The value of results looks like:')
print(y)
print('y should look like:')
print(y0)
max_abs_residual=max ([abs (a-b) for a,b in zip(y,y0)])
print(f'Residual (infinity norm): {max_abs_residual}')
assert max_abs_residual<=1e-14,"residual={} which is greater than 1e-14 threshold".format(max_abs_residual)

Here is what the input data looks like:
A=defaultdict(<function sparse_matrix.<locals>.<lambda> at 0x7f4163559940>, {0: defaultdict(<class 'float'>, {1: -2.5, 2: 1.2}), 1: defaultdict(<class 'float'>, {0: 0.1, 1: 1.0}), 2: defaultdict(<class 'float'>, {0: 6.0, 1: -1.0})})
x=[1.0, 2.0, 3.0]
The value of results looks like:
[-1.4000000000000004, 2.1, 4.0]
y should look like:
[-1.4, 2.1, 4.0]
Residual (infinity norm): 4.440892098500626e-16


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for spmv (exercise 0). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [16]:
### Test Cell - Exercise 0  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=spmv,
              ex_name='spmv',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to spmv did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

spmv test ran 100 iterations in 0.13 seconds
Passed! Please submit.


### Exercise 1: (3 points)
**edgestosparse**  

**Your task:** define `edgestosparse` as follows:

Create a function to convert edges into a sparse matrix $G$ where %G_st = 1.0$ where an edge $st$ exists.

**Inputs**:
- `edges`: pandas DataFrame representing the graph
- `name2id`: dict where key is a name and id is the value
- `num_rows`: int default None

**Return**: G
- `G`: sparse_matrix technically a collections.defualtdict

**Hints**
- Given `id2name` and `name2id` as computed above, convert `edges` into a sparse matrix, `G`, where there is an entry `G[s][t] == 1.0` wherever an edge `(s, t)` exists.
- There are multiple ways to solve
- This could take a while for the kernel to process as there are 1 million rows


In [17]:
### Solution - Exercise 1  
def edgestosparse(edges: pd.DataFrame, name2id: dict, num_rows:int=None) -> collections.defaultdict:
    G = sparse_matrix() #iitialize an empty sparse matrix
    
     # Iterate through each row of the DataFrame
    for index, row in edges.iterrows():
        #columns are names "Source" and "Target"
        #get ids for names
        s = name2id[row['Source']] #name2id: dict where key is a name and id is the value
        t = name2id[row['Target']]
            
        #add to the sparse matrix to create G[s][t] = 1.0
        G[s][t] = 1.0
    
    return G

### Demo function call
from random import sample
G=edgestosparse(edges, name2id)
print('Number of expected verticies and edges')
print(f'number of vertices={num_verts}\nnumber of edges={num_edges}\n')
print('edgestoparse results looks like:')
print(f'number of vertices={len(G.keys())}\nnumber of edges={sum([len(row_i) for row_i in G.values()])}\n')

# Check a random sample
for k in sample(range(num_edges), 1000):
    i = name2id[edges['Source'].iloc[k]]
    j = name2id[edges['Target'].iloc[k]]
    assert i in G, "source {} `i` not found in `G`".format(i)
    assert j in G[i], "target {} `j` not found in `G[i]`".format(j)
    assert G[i][j] == 1.0, "for source {} and target {}, `G[i][j]`={}!=1.0".format(i,j,G[i][j])
print(f'Checked random sample')

Number of expected verticies and edges
number of vertices=107456
number of edges=882640

edgestoparse results looks like:
number of vertices=107456
number of edges=882640

Checked random sample


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for edgestosparse (exercise 1). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [18]:
### Test Cell - Exercise 1  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=edgestosparse,
              ex_name='edgestosparse',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to edgestosparse did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

edgestosparse test ran 100 iterations in 1.29 seconds
Passed! Please submit.


### RUN ME!!!
Whether your solution is working or not, run the following code cell. It will load the proper results into memory.

In [19]:
### RUN ME!!!
with open('resource/asnlib/publicdata/G.dill', 'rb') as fp:
    G = dill.load(fp)

### Exercise 2: (3 points)
**verticestosparse**  

**Your task:** define `verticestosparse` as follows:

Create a function to construct a sparse matrix which uses vertex names as keys instead of integers.

**Inputs**:
- `edges`: pandas DataFrame representing the graph

**Return**: H
- `H`: sparse_matrix technically a collections.defualtdict

**Hints**
- There are multiple ways to solve
- This could take a while for the kernel to process as there are 1 million rows


In [20]:
### Solution - Exercise 2  
def verticestosparse(edges: pd.DataFrame, num_rows:int=None) -> collections.defaultdict:
    H = sparse_matrix()
    
 
    for index, row in edges.iterrows():
     #columns are names "Source" and "Target"
       #Get the names directly from the columns
        s = row['Source']
        t = row['Target']
        H[s][t] = 1
    
    return H

### Demo function call
from random import sample
H=verticestosparse(edges)
print('Number of expected verticies and edges')
print(f'number of vertices={num_verts}\nnumber of edges={num_edges}\n')
print('verticestosparse results looks like:')
print(f'number of vertices={len(H.keys())}\nnumber of edges={sum([len(row_i) for row_i in H.values()])}\n')

# Check a random sample
for i in sample(G.keys(), 1000):
    i_name = id2name[i]
    assert i_name in H,'`i_name` not in `H`'
    assert len(G[i]) == len(H[i_name]),'`len(G[i])` != `len(H[i_name])`'
print(f'Checked random sample')

Number of expected verticies and edges
number of vertices=107456
number of edges=882640

verticestosparse results looks like:
number of vertices=107456
number of edges=882640

Checked random sample


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for verticestosparse (exercise 2). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [21]:
### Test Cell - Exercise 2  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=verticestosparse,
              ex_name='verticestosparse',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to verticestosparse did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

verticestosparse test ran 100 iterations in 1.43 seconds
Passed! Please submit.


### RUN ME!!!
Whether your solution is working or not, run the following code cell. It will load the proper results into memory.

In [22]:
### RUN ME!!!
with open('resource/asnlib/publicdata/H.dill', 'rb') as fp:
    H = dill.load(fp)

### Exercise 3: (3 points)
**spmv_keyed**  

**Your task:** define `spmv_keyed` as follows:

Create a function to compute $y \leftarrow A x$ for matricies with named keys.

**Inputs**:
- `A`: sparse_matrix technically a collections.defaultdict
- `x`: dense_vector technically a list

**Return**: Return $y \leftarrow A x$
- `y`: dense_vector technically a dict

**Hints**
- Recall you did a matrix-vector multiply in Notebook 10 Part 2. Adapt that logic for when `A` is now a sparse matrix.
- Conceptually, for all $i$, $y_{i} = \sum ^{j} A_{ij}x_{j}$
- Go back to Exercise 0. If you implemented it well, a modest change will likely be all you need.


In [23]:
### Solution - Exercise 3  
def spmv_keyed(A: collections.defaultdict, x: list) -> dict:
    assert type(x) is dict
    y = vector_keyed(keys=A.keys(), values=0.0)
    #y = A*x
    #A deaultdict mapping row/col to values 
    #Iterate over sparse matrix A as nested dictionary
     #Iterate over sparse matrix A as nested dictionary
    for row_name, row_data in A.items(): #Iterate over each row index 'i' in the matrix
        for col_name, value in row_data.items(): # loop through column in each row {column_index: value}

            y[row_name] += value * x.get(col_name, 0) #Find the value in your vector  for that target name

    return y

### Demo function call
#   'row':  / 0.   -2.5   1.2 \   / 1. \   / -1.4 \
#  'your':  | 0.1   1.    0.  | * | 2. | = |  2.1 |
#  'boat':  \ 6.   -1.    0.  /   \ 3. /   \  4.0 /
keys=['row','your','boat']
Akeyed = sparse_matrix()
Akeyed['row']['your'] = -2.5
Akeyed['row']['boat'] = 1.2
Akeyed['your']['row'] = 0.1
Akeyed['your']['your'] = 1.
Akeyed['boat']['row'] = 6.
Akeyed['boat']['your'] = -1.

xkeyed = vector_keyed (keys, [1, 2, 3])
y0keyed = vector_keyed (keys,[-1.4, 2.1, 4.0])

print('Here is what the input data looks like:')
print(f'Akeyed={Akeyed}\nxkeyed={xkeyed}')
ykeyed = spmv_keyed(Akeyed, xkeyed)
print('The value of results looks like:')
print(ykeyed)
print('y should look like:')
print(y0keyed)
max_abs_residual=max ([abs (r) for r in [(ykeyed[k] - y0keyed[k]) for k in keys]])
print(f'Residual (infinity norm): {max_abs_residual}')
assert max_abs_residual<=1e-14,"residual={} which is greater than 1e-14 threshold".format(max_abs_residual)

Here is what the input data looks like:
Akeyed=defaultdict(<function sparse_matrix.<locals>.<lambda> at 0x7f415b321af0>, {'row': defaultdict(<class 'float'>, {'your': -2.5, 'boat': 1.2}), 'your': defaultdict(<class 'float'>, {'row': 0.1, 'your': 1.0}), 'boat': defaultdict(<class 'float'>, {'row': 6.0, 'your': -1.0})})
xkeyed={'row': 1.0, 'your': 2.0, 'boat': 3.0}
The value of results looks like:
{'row': -1.4000000000000004, 'your': 2.1, 'boat': 4.0}
y should look like:
{'row': -1.4, 'your': 2.1, 'boat': 4.0}
Residual (infinity norm): 4.440892098500626e-16


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for spmv_keyed (exercise 3). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [24]:
### Test Cell - Exercise 3  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=spmv_keyed,
              ex_name='spmv_keyed',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to spmv_keyed did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

spmv_keyed test ran 100 iterations in 0.09 seconds
Passed! Please submit.


Let's time `spmv()` and `spmv_keyed()` on the full data set. Do they perform differently?

> If this benchmark or any of the subsequent ones take an excessively long time to run, including autograder failure, then it's likely you have not implemented `spmv_keyed` efficiently. You'll need to look closely and ensure your implementation is not doing something that leads to excessive running time or memory consumption, which might happen if you don't really understand how dictionaries work.

In [25]:
x = dense_vector ([1.] * num_verts)
times=timeit.repeat('spmv(G, x)',setup='from __main__ import spmv,G,x',number=1,repeat=7)
print(f'spmv function Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')


xkeyed = vector_keyed (keys=[v for v in V_names], values=1.)
times=timeit.repeat('spmv_keyed(H, xkeyed)',setup='from __main__ import spmv_keyed,H,xkeyed',number=1,repeat=7)
print(f'spmv_keyed function Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')
# benchit('spmv(G, x)', scope=globals());
# times=timeit.repeat('scale_colwise(A_rowmaj)',setup='from __main__ import scale_colwise,A_rowmaj',number=1,repeat=7)
# print(f'row-major Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')

spmv function Average 0.241861 seconds with Standard Deviation 0.002383 seconds

spmv_keyed function Average 0.414654 seconds with Standard Deviation 0.013965 seconds



### Alternative Formats
Take a look at the following [slides](https://www.dropbox.com/s/4fwq21dy60g4w4u/cse6040-matrix-storage-notes.pdf?dl=0). These slides cover the basics of two list-based sparse matrix formats known as _coordinate format_ (COO) and _compressed sparse row_ (CSR). We will also discuss them briefly below.

**Coordinate Format (COO)**
In this format we store three lists, one each for rows, columns and the elements of the matrix. Look at the below picture to understand how these lists are formed.

<img src="resource/asnlib/publicdata/coo.png" width="1280">

### Exercise 4: (3 points)
**edgestocoo**  

**Your task:** define `edgestocoo` as follows:

Create a function to convert edges into a COO data structure and return a tuple(coo_rows,coo_cols,coo_vals)

**Inputs**:
- `edges`: pandas DataFrame representing the graph
- `name2id`: dict where key is a name and id is the value

**Return**: tuple(coo_rows,coo_cols,coo_vals)
- `coo_rows`: list representing row indicies for the source verticies
- `coo_cols`: list representing col indicies for the target verticies
- `coo_vals`: list representing weight of the edges

**Hints**
- Think of what rows, columns, and values mean conceptually when you relate it with our dataset of edges
- Similar to previous exercises, our weights should be of value=1.0.


In [26]:
### Solution - Exercise 4  
def edgestocoo(edges: pd.DataFrame, name2id: dict) -> tuple:
   
    coo_rows = edges['Source'].map(name2id).tolist()
    coo_cols = edges['Target'].map(name2id).tolist()
    coo_vals = [1.0] * len(edges) #create a list of 1.0 for each edge
    
    return (coo_rows, coo_cols, coo_vals)

   
### Demo function call
from random import sample
coo_rows,coo_cols,coo_vals=edgestocoo(edges, name2id)
print('Expected lengths of COO variables')
print(f'coo_rows={num_edges}\ncoo_cols={num_edges}\ncoo_vals={num_edges}')
print('Actual lengths of COO variables')
print(f'coo_rows={len(coo_rows)}\ncoo_cols={len(coo_cols)}\ncoo_vals={len(coo_vals)}')

print('Expected all values of coo_vals==1.')
print(f'Actual: all values of coo_vals==1.: {all([v==1. for v in coo_vals])}')

# Check a random sample
coo_zip=zip(coo_rows,coo_cols,coo_vals)
for i,j,a_ij in sample(list(coo_zip), 1000):
    assert (i in G) and (j in G[i]),'for row {} col {}, both should be TRUE `(i in G)` AND `(j in G[i])` : {} AND {}'.format(i,j,(i in G),(j in G[i]))
print(f'Checked random sample')

Expected lengths of COO variables
coo_rows=882640
coo_cols=882640
coo_vals=882640
Actual lengths of COO variables
coo_rows=882640
coo_cols=882640
coo_vals=882640
Expected all values of coo_vals==1.
Actual: all values of coo_vals==1.: True
Checked random sample


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for edgestocoo (exercise 4). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [27]:
### Test Cell - Exercise 4  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=edgestocoo,
              ex_name='edgestocoo',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to edgestocoo did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

edgestocoo test ran 100 iterations in 1.77 seconds
Passed! Please submit.


### RUN ME!!!
Whether your solution is working or not, run the following code cell. It will load the proper results into memory.

In [31]:
### RUN ME!!!
with open('resource/asnlib/publicdata/coo.dill', 'rb') as fp:
    coo_rows, coo_cols, coo_vals = dill.load(fp)

### Exercise 5: (3 points)
**spmv_coo**  

**Your task:** define `spmv_coo` as follows:

Create a function to compute $y \leftarrow A x$ for a COO implementation.

**Inputs**:
- `R`: list representing coo_rows
- `C`: list representing coo_cols
- `V`: list representing coo_vals
- `x`: dense_vector technically a list
- `num_rows`: int default None

**Return**: Return $y \leftarrow A x$
- `y`: dense_vector technically a list

**Hints**
- Recall you did a matrix-vector multiply in Notebook 10 Part 2. Adapt that logic for when `A` is now a sparse matrix.
- Conceptually, for all $i$, $y_{i} = \sum ^{j} A_{ij}x_{j}$


In [29]:
### Solution - Exercise 5  
def spmv_coo(R: list, C: list, V: list, x: list, num_rows:int=None) -> list:
    assert type(x) is list
    assert type(R) is list
    assert type(C) is list
    assert type(V) is list
    assert len(R) == len(C) == len(V)
    if num_rows is None:
        num_rows = max(R) + 1
    
    y = [0.0] * num_rows #initialize the result vector with zeros
        
    for k in range(len(V)): #Process each non-zero entry in the sparse matrix
        row = R[k] #find row and col indices
        col = C[k]
        val = V[k]
        
        y[row] += val * x[col] # # y[row] = sum(A[row, col] * x[col]) #Accumulate the product into the correct row of y
        
    return y

### Demo function call
#   / 0.   -2.5   1.2 \   / 1. \   / -1.4 \
#   | 0.1   1.    0.  | * | 2. | = |  2.1 |
#   \ 6.   -1.    0.  /   \ 3. /   \  4.0 /
A_coo_rows = [0, 0, 1, 1, 2, 2]
A_coo_cols = [1, 2, 0, 1, 0, 1]
A_coo_vals = [-2.5, 1.2, 0.1, 1., 6., -1.]

x_coo = dense_vector([1, 2, 3])
y0_coo = dense_vector([-1.4, 2.1, 4.0])

print('Here is what the input data looks like:')
print(f'A={list(zip(A_coo_rows, A_coo_cols, A_coo_vals))}\nx={x_coo}')
y_coo = spmv_coo(A_coo_rows, A_coo_cols, A_coo_vals, x_coo)
print('The value of results looks like:')
print(y_coo)
print('y should look like:')
print(y0_coo)
max_abs_residual=max ([abs (a-b) for a,b in zip(y_coo,y0_coo)])
print(f'Residual (infinity norm): {max_abs_residual}')
assert max_abs_residual<=1e-14,"residual={} which is greater than 1e-14 threshold".format(max_abs_residual)

Here is what the input data looks like:
A=[(0, 1, -2.5), (0, 2, 1.2), (1, 0, 0.1), (1, 1, 1.0), (2, 0, 6.0), (2, 1, -1.0)]
x=[1.0, 2.0, 3.0]
The value of results looks like:
[-1.4000000000000004, 2.1, 4.0]
y should look like:
[-1.4, 2.1, 4.0]
Residual (infinity norm): 4.440892098500626e-16


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for spmv_coo (exercise 5). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [30]:
### Test Cell - Exercise 5  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=spmv_coo,
              ex_name='spmv_coo',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to spmv_coo did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

spmv_coo test ran 100 iterations in 0.12 seconds
Passed! Please submit.


Similar to above where we timed `spmv()` and `spmv_keyed()`, lets time `spmv_coo()` on the full data set. Does it differ dramatically from the other 2 functions?

In [8]:
xcoo = dense_vector ([1.] * num_verts)
times=timeit.repeat('spmv_coo(coo_rows,coo_cols,coo_vals,xcoo)',setup='from __main__ import spmv_coo,coo_rows,coo_cols,coo_vals,xcoo',number=1,repeat=7)
print(f'spmv_coo function Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')

NameError: name 'dense_vector' is not defined

### Compressed Sparse Row Format

This format tries to compress the sparse matrix further compared to COO format. Suppose you have the following coordinate representation of a sparse matrix where you sort by row index:

```python
    rows   = [0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6]
    cols   = [1, 2, 4, 0, 2, 3, 0, 1, 3, 4, 1, 2, 5, 6, 0, 2, 5, 3, 4, 6, 3, 5]
    values = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
```

If there are `nnz` nonzeroes, then this representation requires `3*nnz` units of storage since it needs three arrays, each of length `nnz`.

Observe that when we sort by row index, values are repeated wherever there is more than one nonzero in a row. The idea behind CSR is to exploit this redundancy. From the sorted COO representation, we keep the column indices and values as-is. Then, instead of storing every row index, we just store the starting _offset_ of each row in those two lists, which we'll refer to as the _row pointers_, stored as the list `rowptr`, below:

```python
    cols   = [1, 2, 4, 0, 2, 3, 0, 1, 3, 4, 1, 2, 5, 6, 0, 2, 5, 3, 4, 6, 3, 5]
    values = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
    rowptr = [0,       3,       6,         10,         14,      17,      20,   22]
```

If the sparse matrix has `n` rows, then the `rowptr` list has `n+1` elements, where the last element (`rowptr[-1] == rowptr[n]`) is `nnz`. (Why might you need this last element?)

### Exercise 6: (3 points)
**coo2csr**  

**Your task:** define `coo2csr` as follows:

Create a function to transform a matrix in COO format to CSR.

**Inputs**:
- `coo_rows`: list representing coo_rows
- `coo_cols`: list representing coo_cols
- `coo_vals`: list representing coo_vals

**Return**: tuple(csr_ptrs,csr_inds,csr_vals)
- `csr_ptrs`: list representing row pointers
- `csr_inds`: list representing column indices
- `csr_vals`: list representing values

**Hints**:
- `csr_inds` and `csr_vals` have already been computed. You just need to derive `csr_ptrs`.


In [32]:
### Solution - Exercise 6  
def coo2csr(coo_rows: list, coo_cols: list, coo_vals: list) -> tuple:
    from operator import itemgetter
    C = sorted(zip(coo_rows, coo_cols, coo_vals), key=itemgetter(0))
    nnz = len(C)
    assert nnz >= 1

    csr_inds = [j for _, j, _ in C] #column pointers
    csr_vals = [a_ij for _, _, a_ij in C]  #value pointers
    
    #csr_ptrs should store the cumulative count (prefix sum) of non-zero elements per row
    #Define num_rows
    num_rows = max(coo_rows) + 1 if coo_rows else 0
    #Initialize csr_ptrs with zeros (length num_rows + 1)
    csr_ptrs = [0] *(num_rows + 1) 
    
    #Count non-zero elements per row
    for r, _, _ in C:
        csr_ptrs[r+1] += 1
         
    for i in range(num_rows): #Compute the prefix sum (cumulative total)
        csr_ptrs[i+1] += csr_ptrs[i]
    
    return (csr_ptrs, csr_inds, csr_vals)

### Demo function call
#Sample Demo
#          B             csr_ptrs
#  / 0 0 0 0 0 0 0 \        0
#  | 0 0 0 0 0 0 0 |        0
#  | 1 0 0 1 0 1 0 |        0
#  | 0 0 0 0 0 0 0 |   -->  3
#  | 0 0 0 0 0 0 0 |        3
#  | 0 1 0 0 1 0 1 |        3
#  \ 0 0 1 1 0 0 0 /        6
#                           8
B_coo_rows = [2, 2, 2, 5, 5, 5, 6, 6]
B_coo_cols = [0, 3, 5, 1, 4, 6, 2, 3] #503
B_coo_vals = [1, 1, 1, 1, 1, 1, 1, 1]
B_csr_ptrs = [0, 0, 0, 3, 3, 3, 6, 8]

your_csr_ptrs, _, _ = coo2csr(B_coo_rows, B_coo_cols, B_coo_vals)

print('Here is what the input data looks like:')
print(f'coo_rows={B_coo_rows}\ncoo_cols={B_coo_cols}\ncoo_vals={B_coo_vals}\n')
print('The csr_ptrs of your function results looks like:')
print(your_csr_ptrs)
print('csr_ptrs should look like:')
print(B_csr_ptrs)
assert your_csr_ptrs == B_csr_ptrs

#Demo using graph network
csr_ptrs, csr_inds, csr_vals = coo2csr(coo_rows, coo_cols, coo_vals)
print("\n\n##############\nDemo Case 2\n")
print('Here is some of the input data looks like:')
print(f'coo_rows={coo_rows[:20]}\ncoo_cols={coo_cols[:20]}\ncoo_vals={coo_vals[:20]}\n')

#check types
print('Expected type of CSR variables')
print(f'csr_ptrs=list\ncsr_inds=list\ncsr_vals=list')
print('Actual type of CSR variables')
print(f'csr_ptrs={type(csr_ptrs)}\ncsr_inds={type(csr_inds)}\ncsr_vals={type(csr_vals)}')

assert type(csr_ptrs) is list, "`csr_ptrs` is not a list."
assert type(csr_inds) is list, "`csr_inds` is not a list."
assert type(csr_vals) is list, "`csr_vals` is not a list."

#check length
print('Expected length of CSR variables')
print(f'csr_ptrs={(num_verts + 1)}\ncsr_inds={num_edges}\ncsr_vals={num_edges}')
print('Actual length of CSR variables')
print(f'csr_ptrs={len(csr_ptrs)}\ncsr_inds={len(csr_inds)}\ncsr_vals={len(csr_vals)}')

assert len(csr_ptrs) == (num_verts + 1), "`csr_ptrs` has {} values instead of {}".format(len(csr_ptrs), num_verts+1)
assert len(csr_inds) == num_edges, "`csr_inds` has {} values instead of {}".format(len(csr_inds), num_edges)
assert len(csr_vals) == num_edges, "`csr_vals` has {} values instead of {}".format(len(csr_vals), num_edges)

#check csr_ptrs[num_verts]==num_edges
print(f'Expected index number of verticies should equal number of edges i.e. csr_ptrs[{num_verts}]={num_edges}')
print(f'Actual index number of verticies should equal number of edges i.e. csr_ptrs[{num_verts}]={csr_ptrs[num_verts]}')
assert csr_ptrs[num_verts] == num_edges, "`csr_ptrs[{}]` == {} instead of {}".format(num_verts, csr_ptrs[num_verts], num_edges)

# Check some random entries
from random import sample
for i in sample(range(num_verts), 1000):
    assert i in G, "`i` not in `G`"
    a, b = csr_ptrs[i], csr_ptrs[i+1]
    msg_prefix = "Row {} should have these nonzeros: {}".format(i, G[i])
    assert (b-a) == len(G[i]), "{}, which is {} nonzeros; instead, it has just {}.".format(msg_prefix, len(G[i]), b-a)
    assert all([(j in G[i]) for j in csr_inds[a:b]]), "{}. However, it may have missing or incorrect column indices: csr_inds[{}:{}] == {}".format(msg_prefix, a, b, csr_inds[a:b])
    assert all([(j in csr_inds[a:b] for j in G[i].keys())]), "{}. However, it may have missing or incorrect column indices: csr_inds[{}:{}] == {}".format(msg_prefix, a, b, csr_inds[a:b])

Here is what the input data looks like:
coo_rows=[2, 2, 2, 5, 5, 5, 6, 6]
coo_cols=[0, 3, 5, 1, 4, 6, 2, 3]
coo_vals=[1, 1, 1, 1, 1, 1, 1, 1]

The csr_ptrs of your function results looks like:
[0, 0, 0, 3, 3, 3, 6, 8]
csr_ptrs should look like:
[0, 0, 0, 3, 3, 3, 6, 8]


##############
Demo Case 2

Here is some of the input data looks like:
coo_rows=[0, 0, 1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
coo_cols=[23765, 86899, 9975, 88815, 93751, 2317, 2355, 3271, 6195, 6466, 8683, 13524, 13911, 14527, 16527, 17831, 19360, 21440, 24209, 25838]
coo_vals=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Expected type of CSR variables
csr_ptrs=list
csr_inds=list
csr_vals=list
Actual type of CSR variables
csr_ptrs=<class 'list'>
csr_inds=<class 'list'>
csr_vals=<class 'list'>
Expected length of CSR variables
csr_ptrs=107457
csr_inds=882640
csr_vals=882640
Actual length of CSR variables
csr_ptrs=107457
csr_inds=882640
csr_vals=882640

 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for coo2csr (exercise 6). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [ ]:
### Test Cell - Exercise 6  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=coo2csr,
              ex_name='coo2csr',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to coo2csr did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

### RUN ME!!!
Whether your solution is working or not, run the following code cell. It will load the proper results into memory.

In [33]:
### RUN ME!!!
with open('resource/asnlib/publicdata/csr.dill', 'rb') as fp:
    csr_ptrs, csr_inds, csr_vals = dill.load(fp)

### Exercise 7: (3 points)
**spmv_csr**  

**Your task:** define `spmv_csr` as follows:

Create a function to compute $y \leftarrow A x$ for a CSR implementation.

**Inputs**:
- `ptr`: list representing CSR row pointers
- `ind`: list representing CSR column indices
- `val`: list representing CSR values
- `x`: dense_vector technically a list
- `num_rows`: int default None

**Return**: Return $y \leftarrow A x$
- `y`: dense_vector technically a list

**Hints**
- Conceptually, for all $i$, $y_{i} = \sum ^{j} A_{ij}x_{j}$


In [37]:
### Solution - Exercise 7  
def spmv_csr(ptr: list, ind: list, val: list, x: list, num_rows:int=None) -> list:
    assert type(ptr) == list
    assert type(ind) == list
    assert type(val) == list
    assert type(x) == list
    if num_rows is None: num_rows = len(ptr) - 1
    assert len(ptr) >= (num_rows+1)
    assert len(ind) >= ptr[num_rows]
    assert len(val) >= ptr[num_rows]
    
    y = dense_vector(num_rows)
     
    for i in range(num_rows): #loop through each row and find non-zeros for that specific row
        start = ptr[i]
        end = ptr[i + 1]
        
    #compute the dot product for this row
        row_sum = 0
        for k in range(start,end):
            col = ind[k]
            v = val[k]
            
            row_sum += v * x[col]
            
            y[i] = row_sum
        
    return y


### Demo function call
#   / 0.   -2.5   1.2 \   / 1. \   / -1.4 \
#   | 0.1   1.    0.  | * | 2. | = |  2.1 |
#   \ 6.   -1.    0.  /   \ 3. /   \  4.0 /

A_csr_ptrs = [ 0,        2,       4,       6]
A_csr_cols = [ 1,   2,   0,   1,  0,   1]
A_csr_vals = [-2.5, 1.2, 0.1, 1., 6., -1.]

x = dense_vector([1, 2, 3])
y0 = dense_vector([-1.4, 2.1, 4.0])

print('Here is what the input data looks like:')
print(f'csr_ptrs={A_csr_ptrs}\ncsr_cols={A_csr_cols}\ncsr_vals={A_csr_vals}\nx={x}')
y_csr = spmv_csr(A_csr_ptrs, A_csr_cols, A_csr_vals, x)
print('The value of results looks like:')
print(y_csr)
print('y should look like:')
print(y0)
max_abs_residual = max([abs (a-b) for a,b in zip(y, y0)])
print(f'Residual (infinity norm): {max_abs_residual}')
assert max_abs_residual <= 1e-14,"residual={} which is greater than 1e-14 threshold".format(max_abs_residual)

Here is what the input data looks like:
csr_ptrs=[0, 2, 4, 6]
csr_cols=[1, 2, 0, 1, 0, 1]
csr_vals=[-2.5, 1.2, 0.1, 1.0, 6.0, -1.0]
x=[1.0, 2.0, 3.0]
The value of results looks like:
[-1.4000000000000004, 2.1, 4.0]
y should look like:
[-1.4, 2.1, 4.0]
Residual (infinity norm): 4.440892098500626e-16


 
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for spmv_csr (exercise 7). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [38]:
### Test Cell - Exercise 7  

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    execute_tests = dill.load(f)

# Execute test
passed, test_case_vars = execute_tests(func=spmv_csr,
              ex_name='spmv_csr',
              key=b'w5UoyOdrOUfsGf6iT4RzFh0CSauPgb1_shbFm-Xx7j8=', 
              n_iter=100)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars

assert passed, 'The solution to spmv_csr did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###
print('Passed! Please submit.')

spmv_csr test ran 100 iterations in 0.16 seconds
Passed! Please submit.


Similar to above where we timed `spmv()` and `spmv_keyed()` and `spmv_coo()`, lets time `spmv_csr` on the full data set. Does it differ dramatically from the other functions?


In [ ]:
xcsr = dense_vector([1.] * num_verts)
times=timeit.repeat('spmv_csr(csr_ptrs, csr_inds, csr_vals, xcsr)',setup='from __main__ import spmv_csr,csr_ptrs,csr_inds,csr_vals,xcsr',number=1,repeat=7)
print(f'spmv_csr function Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')

### Scipy's Implementations
What you should have noticed is that the list-based COO and CSR formats do not really lead to sparse matrix-vector multiply implementations that are much faster than the dictionary-based methods. Let's instead try Scipy's native COO and CSR implementations.

In [39]:
import numpy as np
import scipy.sparse as sp

A_coo_sp = sp.coo_matrix((coo_vals, (coo_rows, coo_cols)))
A_csr_sp = A_coo_sp.tocsr() # Alternatively: sp.csr_matrix((val, ind, ptr))
x_sp = np.ones(num_verts)

times=timeit.repeat('A_coo_sp.dot(x_sp)',setup='from __main__ import A_coo_sp,x_sp',number=1,repeat=7)
print(f'COO in Scipy: Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')

times=timeit.repeat('A_csr_sp.dot(x_sp)',setup='from __main__ import A_csr_sp,x_sp',number=1,repeat=7)
print(f'CSR in Scipy: Average {np.mean(times):.6f} seconds with Standard Deviation {np.std(times):.6f} seconds\n')

COO in Scipy: Average 0.002538 seconds with Standard Deviation 0.000281 seconds

CSR in Scipy: Average 0.002201 seconds with Standard Deviation 0.000278 seconds



### Fin
Please submit this part.